# 02. Limpieza de datos

Esta notebook transforma el survey raw de Stack Overflow en una sola salida final: un dataset limpio, reducido y listo para analisis, visualizacion y dashboarding.

## Alcance

- cargar el dataset survey base
- perfilar faltantes y consistencia general
- estandarizar categorias con problemas visibles
- imputar faltantes con reglas diferenciadas
- reducir columnas al foco analitico del proyecto
- exportar una unica salida final en `CSV`


## Subpaso 1. Carga del dataset base

**Proposito:** partir desde una fuente reproducible y dejar definida la unica salida que genera esta notebook.

**Resultado esperado:** acceso a `data/survey_data_updated.csv` y exportacion final a `data/survey_data_cleaned_reduced.csv`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SURVEY_CSV_PATH = DATA_DIR / "survey_data_updated.csv"
FINAL_OUTPUT_PATH = DATA_DIR / "survey_data_cleaned_reduced.csv"

survey_raw = pd.read_csv(SURVEY_CSV_PATH)
SURVEY_CSV_PATH, FINAL_OUTPUT_PATH


## Subpaso 2. Perfil inicial de calidad

**Proposito:** dimensionar faltantes, tamano y complejidad antes de transformar el dataset.

**Resultado:** se obtiene una vista rapida del volumen de filas, columnas y variables mas afectadas por nulos.

**Hallazgos:** las columnas mas incompletas pertenecen sobre todo a preguntas opcionales y bloques perifericos del survey, mientras que las variables clave del relato final se pueden conservar con imputacion y recorte selectivo.


In [ ]:
missing_before = survey_raw.isna().sum().sort_values(ascending=False)
profile_before = pd.DataFrame(
    {
        "dtype": survey_raw.dtypes.astype(str),
        "missing_values": survey_raw.isna().sum(),
        "non_null_count": survey_raw.count(),
    }
)

print("Shape:", survey_raw.shape)
print("Total missing values:", int(survey_raw.isna().sum().sum()))
missing_before.head(15)


## Subpaso 3. Estandarizacion de categorias

**Proposito:** corregir etiquetas que afectan legibilidad y consistencia de las variables categoricas mas visibles.

**Resultado:** `Country` y `EdLevel` quedan con nombres mas compactos y consistentes para las etapas posteriores.


In [ ]:
survey_clean = survey_raw.copy()

country_mapping = {
    "United States of America": "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
}

edlevel_mapping = {
    "Bachelorï¿½s degree (B.A., B.S., B.Eng., etc.)": "Bachelors degree",
    "Masterï¿½s degree (M.A., M.S., M.Eng., MBA, etc.)": "Masters degree",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional degree",
    "Associate degree (A.A., A.S., etc.)": "Associate degree",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary school",
    "Primary/elementary school": "Primary school",
}

survey_clean["Country"] = survey_clean["Country"].replace(country_mapping)
survey_clean["EdLevel"] = survey_clean["EdLevel"].replace(edlevel_mapping)

survey_clean[["Country", "EdLevel"]].head()


## Subpaso 4. Reglas de imputacion

**Proposito:** separar el tratamiento de faltantes segun tipo de variable y uso analitico esperado.

**Resultado:** se definen variables numericas para mediana, variables categoricas estructuradas para moda y variables multiseleccion o libres para `Not specified`.

**Hallazgo metodologico:** `ConvertedCompYearly` tambien se imputa en esta etapa para no perder observaciones utiles en salario, comparaciones y visualizaciones.


In [ ]:
numeric_impute_cols = [
    "CompTotal",
    "WorkExp",
    "JobSatPoints_1",
    "JobSatPoints_4",
    "JobSatPoints_5",
    "JobSatPoints_6",
    "JobSatPoints_7",
    "JobSatPoints_8",
    "JobSatPoints_9",
    "JobSatPoints_10",
    "JobSatPoints_11",
    "JobSat",
    "ConvertedCompYearly",
]

mode_impute_cols = [
    "RemoteWork",
    "EdLevel",
    "OrgSize",
    "PurchaseInfluence",
    "BuildvsBuy",
    "SOVisitFreq",
    "SOAccount",
    "SOPartFreq",
    "SOComm",
    "AISelect",
    "AISent",
    "AIAcc",
    "AIComplex",
    "AIThreat",
    "TBranch",
    "ICorPM",
    "Knowledge_1",
    "Knowledge_2",
    "Knowledge_3",
    "Knowledge_4",
    "Knowledge_5",
    "Knowledge_6",
    "Knowledge_7",
    "Knowledge_8",
    "Knowledge_9",
    "Frequency_1",
    "Frequency_2",
    "Frequency_3",
    "TimeSearching",
    "TimeAnswering",
    "ProfessionalCloud",
    "ProfessionalQuestion",
    "Industry",
    "SurveyLength",
    "SurveyEase",
]

not_specified_cols = [
    "LearnCode",
    "LearnCodeOnline",
    "TechDoc",
    "YearsCode",
    "YearsCodePro",
    "DevType",
    "BuyNewTool",
    "TechEndorse",
    "Country",
    "Currency",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "LanguageAdmired",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "DatabaseAdmired",
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",
    "PlatformAdmired",
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",
    "WebframeAdmired",
    "EmbeddedHaveWorkedWith",
    "EmbeddedWantToWorkWith",
    "EmbeddedAdmired",
    "MiscTechHaveWorkedWith",
    "MiscTechWantToWorkWith",
    "MiscTechAdmired",
    "ToolsTechHaveWorkedWith",
    "ToolsTechWantToWorkWith",
    "ToolsTechAdmired",
    "NEWCollabToolsHaveWorkedWith",
    "NEWCollabToolsWantToWorkWith",
    "NEWCollabToolsAdmired",
    "OpSysPersonal use",
    "OpSysProfessional use",
    "OfficeStackAsyncHaveWorkedWith",
    "OfficeStackAsyncWantToWorkWith",
    "OfficeStackAsyncAdmired",
    "OfficeStackSyncHaveWorkedWith",
    "OfficeStackSyncWantToWorkWith",
    "OfficeStackSyncAdmired",
    "AISearchDevHaveWorkedWith",
    "AISearchDevWantToWorkWith",
    "AISearchDevAdmired",
    "NEWSOSites",
    "SOHow",
    "AIBen",
    "AIToolCurrently Using",
    "AIToolInterested in Using",
    "AIToolNot interested in Using",
    "AINextMuch more integrated",
    "AINextNo change",
    "AINextMore integrated",
    "AINextLess integrated",
    "AINextMuch less integrated",
    "AIEthics",
    "AIChallenges",
    "Frustration",
    "ProfessionalTech",
]

len(numeric_impute_cols), len(mode_impute_cols), len(not_specified_cols)


## Subpaso 5. Aplicacion de limpieza principal

**Proposito:** ejecutar la imputacion y cerrar la limpieza base del dataset.

**Resultado:** el dataset limpio queda sin valores faltantes y con variables clave listas para analisis posterior.

**Hallazgos:** la combinacion de mediana, moda y `Not specified` permite retener la mayor parte de la muestra sin volver opacas las columnas relevantes del proyecto.


In [ ]:
for column in numeric_impute_cols:
    if column in survey_clean.columns:
        survey_clean[column] = survey_clean[column].fillna(survey_clean[column].median())

for column in mode_impute_cols:
    if column in survey_clean.columns:
        mode_series = survey_clean[column].mode(dropna=True)
        fill_value = mode_series.iloc[0] if not mode_series.empty else "Unknown"
        survey_clean[column] = survey_clean[column].fillna(fill_value)

for column in not_specified_cols:
    if column in survey_clean.columns:
        survey_clean[column] = survey_clean[column].fillna("Not specified")

print("Total missing values after cleaning:", int(survey_clean.isna().sum().sum()))
survey_clean[["ConvertedCompYearly", "RemoteWork", "Country"]].head()


## Subpaso 6. Campos derivados de compensacion

**Proposito:** conservar medidas auxiliares de escala y dispersion para comparaciones numericas posteriores.

**Resultado:** se agregan `ConvertedCompYearly_MinMax` y `ConvertedCompYearly_Zscore` al dataset limpio.


In [ ]:
comp_series = survey_clean["ConvertedCompYearly"]
comp_min = comp_series.min()
comp_max = comp_series.max()
comp_mean = comp_series.mean()
comp_std = comp_series.std(ddof=0)

survey_clean = survey_clean.assign(
    ConvertedCompYearly_MinMax=((comp_series - comp_min) / (comp_max - comp_min)) if comp_max != comp_min else 0.0,
    ConvertedCompYearly_Zscore=((comp_series - comp_mean) / comp_std) if comp_std else 0.0,
).copy()

survey_clean[["ConvertedCompYearly", "ConvertedCompYearly_MinMax", "ConvertedCompYearly_Zscore"]].head()


## Subpaso 7. Reduccion al dataset final

**Proposito:** conservar solo las columnas que sostienen el relato principal del caso: perfil de la muestra, experiencia, compensacion, satisfaccion y stack tecnologico.

**Resultado:** se obtiene una version compacta, consistente y lista para consumir directamente en las notebooks siguientes.

**Hallazgos:** el recorte mantiene la misma cantidad de filas y concentra el dataset en las variables que realmente alimentan analisis, visualizaciones y dashboarding.


In [ ]:
focused_reduced_columns = [
    "ResponseId",
    "MainBranch",
    "Age",
    "Employment",
    "RemoteWork",
    "EdLevel",
    "YearsCode",
    "YearsCodePro",
    "DevType",
    "OrgSize",
    "Country",
    "Currency",
    "CompTotal",
    "ConvertedCompYearly",
    "ConvertedCompYearly_MinMax",
    "ConvertedCompYearly_Zscore",
    "WorkExp",
    "Industry",
    "JobSat",
    "JobSatPoints_6",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "LanguageAdmired",
    "DatabaseHaveWorkedWith",
    "DatabaseWantToWorkWith",
    "DatabaseAdmired",
    "PlatformHaveWorkedWith",
    "PlatformWantToWorkWith",
    "PlatformAdmired",
    "WebframeHaveWorkedWith",
    "WebframeWantToWorkWith",
    "WebframeAdmired",
]

focused_reduced_columns = [column for column in focused_reduced_columns if column in survey_clean.columns]
survey_clean_reduced = survey_clean[focused_reduced_columns].copy()

print("Cleaned shape in memory:", survey_clean.shape)
print("Final reduced shape:", survey_clean_reduced.shape)
survey_clean_reduced.head()


## Subpaso 8. Exportacion de la unica salida final

**Proposito:** guardar un unico dataset final que centralice todo el trabajo de limpieza.

**Resultado:** esta notebook produce solamente `data/survey_data_cleaned_reduced.csv`.

**Hallazgo operativo:** dejar una sola salida evita ambiguedad entre versiones y simplifica el resto del flujo analitico.


In [ ]:
survey_clean_reduced.to_csv(FINAL_OUTPUT_PATH, index=False)
print("Saved:", FINAL_OUTPUT_PATH)


## Cierre de la etapa

La limpieza termina con una sola salida reutilizable y lista para el resto del proyecto. La siguiente notebook ya no necesita exportar tablas auxiliares: puede concentrarse directamente en EDA, estadisticas y visualizaciones sobre esta version final.
